# External pose estimators flow

Runs external robot pose estimators (RoboPEPP, HPE, or the ground-truth reference `dream_ground_truth`; see `DESIGN-Import-Competitors.md`) on a group of the demonstrations of a pack, verifies each, and compares them.

For every run of `external_runs` (a checked-in run of `external_proprioception`), the flow writes a copy that runs on the `eval_group` of the pack, and queues its `Run_External_Proprioception.ipynb` and `Verify_VisualProprioception.ipynb`. The comparison uses the robot run `compare_robot`: the predictions of every method are mapped onto its joints by name, so methods predicting different joints (RoboPEPP 1-6, HPE 1-7) are compared on the joints they share.

The demonstrations must come from a source sequence that the methods support (`source_sequence` in their metadata), e.g. `panda-dream-freeform`, from the DREAM RealSense sequence.

In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import yaml
from flow import setup_flow, flow_entry, run_flow, display_flow_report
from demonstration.demopack import import_demopack, group_chooser_sp_vp_standard, group_chooser_fixed_order

## Parameters

In [ ]:
flow_name = "External_flow_00"
pack = "panda-dream-freeform"
camera = "dev000"
# checked-in runs of external_proprioception, e.g. "robopepp_panda_published", "hpe_panda_published"
external_runs = ["dream_ground_truth_panda"]
# the robot run whose joints the comparison reports (panda_6dof: the joints of the DREAM papers)
compare_robot = ["robot_franka", "panda_6dof"]
# the order of the demonstrations for the split; None for the sorted order
demo_order = None
# the group of the pack the methods run on: vp_testing is held out from our training
eval_group = "vp_testing"
creation_style = "exist-ok"

## Workspace, demonstrations and the exp/runs

In [ ]:
expruns_path, results_path, notebooks_path = setup_flow(flow_name, [
    "demonstration", "robot_franka", "robot_g1", "robot_inspire_hand",
    "external_proprioception", "visual_proprioception", "visual_proprioception_collections"])

if demo_order is None:
    group_chooser = group_chooser_sp_vp_standard
else:
    group_chooser = group_chooser_fixed_order(demo_order, group_chooser_sp_vp_standard)
selection = import_demopack(pathlib.Path(Config()["demopacks_path"], pack).expanduser(), group_chooser)
validation_data = [[pack, demo, camera] for demo in selection[eval_group]]
print(f"{eval_group}: {validation_data}")

def write_exprun(exp_name, run_name, val):
    path = pathlib.Path(Config().get_exprun_path(), exp_name, run_name + ".yaml")
    with open(path, "w") as f:
        yaml.dump(val, f)

expruns = []
runs = []
for template in external_runs:
    with open(pathlib.Path(expruns_path, "external_proprioception", template + ".yaml")) as f:
        val = yaml.safe_load(f)
    run = f"{template}_{eval_group}"
    val.update({"name": run, "validation_data": validation_data})
    write_exprun("external_proprioception", run, val)
    runs.append(run)
    expruns.append(flow_entry(f"Run_{run}", "external_proprioception", run, 0, creation_style))
    expruns.append(flow_entry(f"Verify_{run}", "external_proprioception", run, 1, "exist-ok"))

compare_run = f"external_{pack}_{eval_group}"
write_exprun("visual_proprioception_collections", compare_run, {
    "name": compare_run, "source_experiment": "external_proprioception", "tocompare": runs,
    "robot_exp": compare_robot[0], "robot_run": compare_robot[1],
    "input-to-notebook": ["visual_proprioception/Compare_VisualProprioception.ipynb"]})
expruns.append(flow_entry(f"Compare_{compare_run}", "visual_proprioception_collections", compare_run, 0, creation_style))
print(f"{len(expruns)} stages")

In [ ]:
flow_error = None
try:
    run_flow(expruns, expruns_path, results_path, notebooks_path)
except Exception as error:
    flow_error = error

In [ ]:
final_results_path = pathlib.Path(results_path, "visual_proprioception_collections", compare_run)
report = display_flow_report(
    expruns, results_path, notebooks_path, final_results_path,
    flow_error=flow_error,
    preview_files=sorted(final_results_path.glob("msecomparison*.png")))
if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")